# 🦊 Bank Soal & Latihan: `ColumnTransformer` (Scikit-Learn)

Selamat datang, calon *ML Master*! 🎉
Di notebook ini kamu akan **praktik langsung** memakai `ColumnTransformer` lewat **3 studi kasus** yang makin lama makin seru.

| Bagian | Isi | Level |
|---|---|---|
| **A** | Kuis teori (8 soal) | 🟢 Pemanasan |
| **B** | Studi Kasus 1 — Prediksi Harga Rumah 🏠 | 🟢 Mudah |
| **C** | Studi Kasus 2 — Prediksi Pelanggan Berhenti Langganan 📺 | 🟡 Menengah |
| **D** | Kunci Jawaban 🔑 | Buka kalau sudah mencoba! |
| **E** | Klaim Badge 🏆 | Reward |

### 📌 Cara pakai
1. Jalankan sel dari atas ke bawah (**Runtime → Run all** tidak disarankan, karena ada bagian yang harus kamu isi).
2. Cari tanda `___` di sel kode — itu bagian yang **harus kamu isi**.
3. Setelah mengisi, jalankan sel **"✅ Cek jawabanmu"** di bawahnya. Kalau muncul ✅ berarti benar!
4. Mentok? Buka **Petunjuk** dulu. Masih mentok? Baru intip **Kunci Jawaban** di Bagian E.

> 💬 *Semangat! Salah itu wajar. Error adalah cara komputer ngajarin kita.*

## ⚙️ Persiapan

In [ ]:
import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score, classification_report

pd.set_option("display.max_columns", 30)
print("Versi scikit-learn :", sklearn.__version__)
print("Siap belajar!")

---
# 🅰️ Bagian A — Kuis Teori (Pemanasan)

Jawab dulu di kepala (atau di kertas), lalu klik **Lihat jawaban**.

---
**Soal 1.** Kenapa kita butuh `ColumnTransformer`?
- a) Supaya model jalan lebih cepat
- b) Supaya kolom yang berbeda jenis bisa diproses dengan cara yang berbeda dalam satu langkah
- c) Supaya data otomatis bersih dari *missing value*
- d) Supaya tidak perlu split data

<details><summary>Lihat jawaban</summary>

**Jawaban: b.** Kolom angka butuh *scaling*, kolom kategori butuh *encoding*, kolom teks butuh *vectorizer*. `ColumnTransformer` membantu kita memberi "perlakuan" yang beda untuk tiap kelompok kolom, lalu menggabungkan hasilnya.
</details>

---
**Soal 2.** Sebuah transformer di `ColumnTransformer` ditulis sebagai tuple dengan 3 isi. Apa urutannya?

<details><summary>Lihat jawaban</summary>

**`(nama, transformer, kolom)`**. Contoh: `("angka", StandardScaler(), ["umur", "gaji"])`.
</details>

---
**Soal 3.** Kolom yang **tidak** disebut di `transformers` akan diapakan secara default? Bagaimana cara mengubahnya?

<details><summary>Lihat jawaban</summary>

Default-nya **dibuang** (`remainder="drop"`). Kalau mau tetap dibawa tanpa diubah, pakai `remainder="passthrough"`.
</details>

---
**Soal 4.** Benar atau Salah? *"Boleh melakukan `StandardScaler().fit()` ke seluruh dataset dulu, baru split train/test, karena toh cuma scaling."*

<details><summary>Lihat jawaban</summary>

**Salah.** Itu namanya **data leakage**: informasi data test (rata-rata & standar deviasi) ikut "bocor" ke proses training. Solusinya: split dulu, lalu masukkan `ColumnTransformer` ke dalam `Pipeline` supaya `fit` hanya dilakukan pada data train.
</details>

---
**Soal 5.** Apa bedanya `Pipeline` dan `ColumnTransformer`?

<details><summary>Lihat jawaban</summary>

- `Pipeline` = langkah **berurutan** (seri): hasil langkah 1 jadi masukan langkah 2.
- `ColumnTransformer` = langkah **paralel**: tiap kelompok kolom diproses sendiri-sendiri, lalu hasilnya **digabung** (concatenate) ke samping.

Biasanya keduanya dipakai bersama. 🤝
</details>

---
**Soal 6.** Apa fungsi `OneHotEncoder(handle_unknown="ignore")`?

<details><summary>Lihat jawaban</summary>

Kalau saat prediksi muncul kategori **yang belum pernah dilihat** saat training (misal kota baru), encoder tidak error. Semua kolom one-hot untuk kategori itu diisi 0.
</details>

---
**Soal 7.** Kamu ingin memproses kolom teks `"ulasan"` dengan `TfidfVectorizer`. Mana yang benar?
- a) `("teks", TfidfVectorizer(), ["ulasan"])`
- b) `("teks", TfidfVectorizer(), "ulasan")`

<details><summary>Lihat jawaban</summary>

**Jawaban: b.** `TfidfVectorizer` butuh input **1 dimensi** (satu deret teks), jadi nama kolom ditulis sebagai **string**, bukan list. Kalau pakai list `["ulasan"]`, scikit-learn mengirim DataFrame 2 dimensi, dan hasilnya bisa **error** atau (lebih licik!) **diam-diam salah**. Kamu akan membuktikannya di Studi Kasus 3!
</details>

---
**Soal 8.** Setelah `fit`, bagaimana cara melihat nama-nama kolom hasil transformasi?

<details><summary>Lihat jawaban</summary>

Pakai **`preprocessor.get_feature_names_out()`**. Nama kolom diawali nama transformer, misalnya `angka__luas_tanah` atau `kategori__lokasi_Depok`.
</details>

> 🎯 **Skor kamu:** 7–8 benar = Jago! 5–6 = Bagus, lanjut praktik. ≤4 = Santai, baca ulang PDF materi sebentar ya.

---
# 🅱️ Bagian B — Studi Kasus 1: Prediksi Harga Rumah 🏠 (🟢 Mudah)

**Cerita:** Kamu data scientist di startup properti. Tim marketing ingin tahu **estimasi harga rumah** (dalam juta rupiah) dari data lokasi, kondisi, dan ukuran rumah.

**Tantangannya:** datanya campur — ada **angka** dan ada **kategori**, plus ada **data kosong** (*missing value*).

**Yang akan kamu praktikkan:** `ColumnTransformer` dasar + `Pipeline` + `SimpleImputer` + `StandardScaler` + `OneHotEncoder`.

In [ ]:
def buat_data_rumah(n=500, seed=42):
    rng = np.random.default_rng(seed)
    lokasi = rng.choice(["Jakarta Selatan", "Depok", "Bekasi", "Tangerang", "Bogor"], n, p=[.20, .25, .20, .20, .15])
    kondisi = rng.choice(["Baru", "Bagus", "Perlu Renovasi"], n, p=[.25, .50, .25])
    luas_tanah = rng.normal(120, 40, n).clip(40, 300).round()
    luas_bangunan = (luas_tanah * rng.uniform(.5, .9, n)).round()
    jumlah_kamar = np.clip((luas_bangunan / 30 + rng.normal(0, .7, n)).round(), 1, 6)
    umur_rumah = rng.integers(0, 31, n).astype(float)

    harga_per_m2 = {"Jakarta Selatan": 30, "Depok": 12, "Bekasi": 10, "Tangerang": 14, "Bogor": 8}
    faktor_kondisi = {"Baru": 1.15, "Bagus": 1.0, "Perlu Renovasi": 0.8}
    f_lokasi = np.array([harga_per_m2[l] for l in lokasi])
    f_kondisi = np.array([faktor_kondisi[k] for k in kondisi])
    harga = (luas_tanah * f_lokasi + luas_bangunan * 4) * f_kondisi * (1 - 0.01 * umur_rumah)
    harga = (harga + rng.normal(0, 80, n)).round(1)

    df = pd.DataFrame({
        "lokasi": lokasi, "kondisi": kondisi,
        "luas_tanah": luas_tanah, "luas_bangunan": luas_bangunan,
        "jumlah_kamar": jumlah_kamar, "umur_rumah": umur_rumah,
        "harga_juta": harga,
    })
    # sengaja bikin data bolong (missing value) biar realistis
    for kolom, persen in [("jumlah_kamar", .08), ("umur_rumah", .06), ("kondisi", .05)]:
        df.loc[rng.random(n) < persen, kolom] = np.nan
    return df

df_rumah = buat_data_rumah()
df_rumah.to_csv("data_rumah.csv", index=False)   # tersimpan, bisa kamu download dari Colab
print("Ukuran data:", df_rumah.shape)
df_rumah.head()

In [ ]:
# Intip dulu: tipe data & jumlah data kosong
print(df_rumah.dtypes, "\n")
print("Jumlah data kosong per kolom:")
print(df_rumah.isna().sum())

### ✏️ Tugas 1.1 — Pisahkan fitur & target, lalu split data

Target kita adalah kolom `harga_juta`. Simpan fitur di `X`, target di `y`, lalu bagi 80% train dan 20% test dengan `random_state=42`.

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

Gunakan `drop(columns="nama_kolom")` untuk membuang kolom, dan `test_size=0.2` untuk 20%.

</details>

In [ ]:
X = df_rumah.drop(columns=___)
y = df_rumah[___]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=___, random_state=42)

In [ ]:
# ✅ Cek jawabanmu
try:
    assert X_train.shape == (400, 6), f"Bentuk X_train harusnya (400, 6), tapi {X_train.shape}"
    assert "harga_juta" not in X_train.columns, "Kolom target masih ada di X!"
    print("✅ Mantap! Data sudah terbagi dengan benar.")
except Exception as e:
    print("❌ Belum pas:", e)

### ✏️ Tugas 1.2 — Kelompokkan kolom

Masukkan nama kolom ke dua list: `kolom_angka` (kolom bertipe angka) dan `kolom_kategori` (kolom bertipe teks/kategori).

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

Lihat output `dtypes` di atas: `float64` = angka, `object`/`str` = kategori.

</details>

In [ ]:
kolom_angka = [___]
kolom_kategori = [___]

In [ ]:
# ✅ Cek jawabanmu
try:
    assert set(kolom_angka) == {"luas_tanah", "luas_bangunan", "jumlah_kamar", "umur_rumah"}, "Isi kolom_angka belum tepat"
    assert set(kolom_kategori) == {"lokasi", "kondisi"}, "Isi kolom_kategori belum tepat"
    print("✅ Betul! 4 kolom angka dan 2 kolom kategori.")
except Exception as e:
    print("❌ Belum pas:", e)

### ✏️ Tugas 1.3 — Buat 'jalur' (pipeline) untuk tiap jenis kolom

Buat dua pipeline kecil:
- `pipe_angka`: isi data kosong dengan **median**, lalu **StandardScaler**.
- `pipe_kategori`: isi data kosong dengan **modus** (`"most_frequent"`), lalu **OneHotEncoder** yang tidak error saat ketemu kategori baru (`handle_unknown="ignore"`).

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

Strategi imputer: `"median"` untuk angka, `"most_frequent"` untuk kategori.

</details>

In [ ]:
pipe_angka = Pipeline([
    ("imputer", SimpleImputer(strategy=___)),
    ("scaler", ___),
])

pipe_kategori = Pipeline([
    ("imputer", SimpleImputer(strategy=___)),
    ("onehot", OneHotEncoder(handle_unknown=___)),
])

In [ ]:
# ✅ Cek jawabanmu
try:
    assert pipe_angka.named_steps["imputer"].strategy == "median", "Strategi imputer angka harusnya 'median'"
    assert isinstance(pipe_angka.named_steps["scaler"], StandardScaler), "Scaler harusnya StandardScaler()"
    assert pipe_kategori.named_steps["imputer"].strategy == "most_frequent", "Strategi imputer kategori harusnya 'most_frequent'"
    assert pipe_kategori.named_steps["onehot"].handle_unknown == "ignore", "handle_unknown harusnya 'ignore'"
    print("✅ Keren! Dua jalur sudah siap.")
except Exception as e:
    print("❌ Belum pas:", e)

### ✏️ Tugas 1.4 — Gabungkan dengan ColumnTransformer 🎯

Inilah momen utamanya! Gabungkan kedua jalur ke dalam `ColumnTransformer` bernama `preprocessor`, lalu `fit` ke `X_train`.

Format tiap transformer: `(nama, transformer, kolom)`.

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

Isi bagian kedua dengan pipeline yang kamu buat di Tugas 1.3, dan bagian ketiga dengan list kolom dari Tugas 1.2.

</details>

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("angka", ___, ___),
        ("kategori", ___, ___),
    ]
)
preprocessor.fit(X_train)

In [ ]:
# ✅ Cek jawabanmu
try:
    hasil = preprocessor.transform(X_train)
    assert hasil.shape == (400, 12), f"Bentuk hasil harusnya (400, 12), tapi {hasil.shape}"
    assert not np.isnan(np.asarray(hasil)).any(), "Masih ada NaN di hasil!"
    print("✅ Luar biasa! 4 kolom angka + 8 kolom one-hot (5 lokasi + 3 kondisi) = 12 kolom, tanpa NaN.")
except Exception as e:
    print("❌ Belum pas:", e)

### ✏️ Tugas 1.5 — Latih model dan ukur hasilnya

Bungkus `preprocessor` dan `RandomForestRegressor(n_estimators=100, random_state=42)` dalam satu `Pipeline` bernama `model_rumah`. Latih dengan **data train**, prediksi **data test**, lalu hitung `mae` (rata-rata kesalahan, dalam juta rupiah) dan `r2`.

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

`fit` pakai `X_train, y_train`. `predict` pakai `X_test`. Jangan tertukar ya!

</details>

In [ ]:
model_rumah = Pipeline([
    ("prep", ___),
    ("regresor", RandomForestRegressor(n_estimators=100, random_state=42)),
])
model_rumah.fit(___, ___)

pred = model_rumah.predict(___)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
print(f"MAE: {mae:.1f} juta | R2: {r2:.3f}")

In [ ]:
# ✅ Cek jawabanmu
try:
    assert r2 > 0.85, f"R2 baru {r2:.3f}. Cek lagi: apakah model dilatih pakai data TRAIN?"
    print(f"✅ Hebat! R2 = {r2:.3f}. Model kamu sudah paham pola harga rumah.")
except Exception as e:
    print("❌ Belum pas:", e)

### 🧠 Refleksi Kasus 1
Jalankan sel di bawah untuk melihat nama kolom hasil transformasi, lalu jawab di kepala: **Kenapa nama kolomnya diawali `angka__` dan `kategori__`?**

<details><summary>Lihat jawaban</summary>

Awalan itu diambil dari **nama transformer** yang kamu tulis di `ColumnTransformer` (`"angka"` dan `"kategori"`). Itulah gunanya memberi nama yang jelas!
</details>

In [ ]:
# Dijalankan setelah Tugas 1.4 selesai
print(preprocessor.get_feature_names_out())

---
# 🅲 Bagian C — Studi Kasus 2: Prediksi Pelanggan Berhenti Langganan 📺 (🟡 Menengah)

**Cerita:** Kamu bekerja di layanan streaming film. Bos ingin tahu **pelanggan mana yang akan berhenti berlangganan** (*churn*) bulan depan, supaya tim bisa memberi promo tepat sasaran.

**Tantangannya:** ada **4 jenis kolom** yang butuh perlakuan beda:

| Jenis kolom | Contoh | Perlakuan |
|---|---|---|
| Angka | `usia`, `biaya_bulanan` | isi kosong + `StandardScaler` |
| Kategori berurutan (**ordinal**) | `paket`: Basic < Standard < Premium | `OrdinalEncoder` |
| Kategori tanpa urutan (**nominal**) | `metode_bayar`, `perangkat` | `OneHotEncoder` |
| Biner 0/1 | `punya_promo` | dibiarkan apa adanya (`passthrough`) |

**Yang akan kamu praktikkan:** `make_column_selector`, `OrdinalEncoder`, `remainder`, dan *tuning* lewat `GridSearchCV`.

In [ ]:
def buat_data_churn(n=800, seed=7):
    rng = np.random.default_rng(seed)
    paket = rng.choice(["Basic", "Standard", "Premium"], n, p=[.45, .35, .20])
    metode = rng.choice(["Kartu Kredit", "E-Wallet", "Transfer Bank", "Pulsa"], n, p=[.25, .40, .20, .15])
    perangkat = rng.choice(["HP", "TV", "Laptop"], n, p=[.55, .25, .20])
    punya_promo = rng.integers(0, 2, n)

    usia = rng.normal(30, 9, n).clip(17, 65).round()
    lama = rng.integers(1, 49, n).astype(float)
    biaya = np.array([{"Basic": 55, "Standard": 90, "Premium": 140}[p] for p in paket], dtype=float)
    biaya = biaya + rng.normal(0, 4, n)
    jam = rng.gamma(2.2, 3.0, n).clip(0, 40).round(1)

    skor = (-0.10 * lama - 0.22 * jam + 0.020 * biaya
            - 1.4 * punya_promo
            + (np.array(metode) == "Pulsa") * 1.5
            + (paket == "Basic") * 0.9 - (paket == "Premium") * 0.5
            + rng.normal(0, 0.5, n) + 0.8)
    churn = (1 / (1 + np.exp(-skor)) > rng.random(n)).astype(int)

    df = pd.DataFrame({
        "usia": usia, "lama_berlangganan_bulan": lama, "biaya_bulanan": biaya.round(0),
        "jam_nonton_per_minggu": jam, "paket": paket, "metode_bayar": metode,
        "perangkat": perangkat, "punya_promo": punya_promo, "churn": churn,
    })
    for kolom, persen in [("lama_berlangganan_bulan", .05), ("jam_nonton_per_minggu", .07), ("usia", .04)]:
        df.loc[rng.random(n) < persen, kolom] = np.nan
    return df

df_churn = buat_data_churn()
df_churn.to_csv("data_churn.csv", index=False)
print("Ukuran data:", df_churn.shape)
print("Persentase pelanggan churn:", round(df_churn["churn"].mean() * 100, 1), "%")
df_churn.head()

### ✏️ Tugas 2.1 — Siapkan data (pakai stratify!)

Pisahkan `X` dan `y` (target: `churn`). Karena kelas churn tidak seimbang, tambahkan `stratify=y` saat split (test 20%, `random_state=42`).

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

`stratify=y` menjaga proporsi kelas di train dan test tetap mirip.

</details>

In [ ]:
X = df_churn.drop(columns=___)
y = df_churn[___]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=___
)

In [ ]:
# ✅ Cek jawabanmu
try:
    assert X_train.shape == (640, 8), f"X_train harusnya (640, 8), tapi {X_train.shape}"
    selisih = abs(y_train.mean() - y_test.mean())
    assert selisih < 0.01, "Proporsi churn di train & test berbeda jauh. Sudah pakai stratify=y?"
    print("✅ Pas! Proporsi churn di train dan test seimbang.")
except Exception as e:
    print("❌ Belum pas:", e)

### ✏️ Tugas 2.2 — Pilih kolom angka secara otomatis

Capek ngetik nama kolom satu-satu? Pakai `make_column_selector`! Buat `selektor_angka` yang otomatis memilih semua kolom bertipe **float** (`dtype_include="float"`).

Lalu cek hasilnya dengan `selektor_angka(X_train)`.

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

Tulis `dtype_include="float"`. Kolom `punya_promo` bertipe integer sehingga tidak terpilih. Nanti kita urus terpisah.

</details>

In [ ]:
selektor_angka = make_column_selector(dtype_include=___)
print(selektor_angka(X_train))

In [ ]:
# ✅ Cek jawabanmu
try:
    terpilih = selektor_angka(X_train)
    assert set(terpilih) == {"usia", "lama_berlangganan_bulan", "biaya_bulanan", "jam_nonton_per_minggu"}, f"Terpilih: {terpilih}"
    print("✅ Benar! Selector memilih 4 kolom float. (punya_promo bertipe int, jadi tidak ikut.)")
except Exception as e:
    print("❌ Belum pas:", e)

### ✏️ Tugas 2.3 — Siapkan 3 transformer berbeda

Buat tiga transformer:
- `tf_angka`: `Pipeline` → imputer median + StandardScaler
- `tf_ordinal`: `OrdinalEncoder` untuk kolom `paket` dengan urutan **Basic, Standard, Premium** (pakai parameter `categories=[[...]]`)
- `tf_nominal`: `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` (kita set `sparse_output=False` supaya bisa tampil sebagai DataFrame di tugas berikutnya)

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

Urutan dari yang terendah ke tertinggi: Basic → Standard → Premium. Tulis dalam list di dalam list: `[["Basic", "Standard", "Premium"]]`.

</details>

In [ ]:
tf_angka = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

tf_ordinal = OrdinalEncoder(categories=[[___, ___, ___]])

tf_nominal = OneHotEncoder(handle_unknown=___, sparse_output=False)

In [ ]:
# ✅ Cek jawabanmu
try:
    assert list(tf_ordinal.categories[0]) == ["Basic", "Standard", "Premium"], "Urutan kategori ordinal belum benar"
    assert tf_nominal.handle_unknown == "ignore", "handle_unknown harusnya 'ignore'"
    print("✅ Tiga transformer siap tempur!")
except Exception as e:
    print("❌ Belum pas:", e)

### ✏️ Tugas 2.4 — Rakit ColumnTransformer dengan 4 jalur

Gabungkan semuanya:
- `"angka"` → `tf_angka` → pakai `selektor_angka`
- `"ordinal"` → `tf_ordinal` → kolom `["paket"]`
- `"nominal"` → `tf_nominal` → kolom `["metode_bayar", "perangkat"]`
- `"biner"` → `"passthrough"` → kolom `["punya_promo"]`

Atur `remainder="drop"` dan tambahkan `.set_output(transform="pandas")` supaya hasilnya berupa DataFrame yang enak dibaca.

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

Selector boleh langsung dipakai di posisi 'kolom': `("angka", tf_angka, selektor_angka)`. Untuk biner, transformer-nya cukup string `"passthrough"`.

</details>

In [ ]:
preprocessor2 = ColumnTransformer(
    transformers=[
        ("angka", tf_angka, ___),
        ("ordinal", tf_ordinal, ___),
        ("nominal", tf_nominal, ___),
        ("biner", ___, ["punya_promo"]),
    ],
    remainder=___,
).set_output(transform="pandas")

hasil = preprocessor2.fit_transform(X_train)
hasil.head()

In [ ]:
# ✅ Cek jawabanmu
try:
    assert hasil.shape == (640, 13), f"Bentuk harusnya (640, 13), tapi {hasil.shape}"
    assert hasil.isna().sum().sum() == 0, "Masih ada NaN"
    assert "ordinal__paket" in hasil.columns, "Kolom ordinal__paket tidak ditemukan"
    print("✅ Sempurna! 4 angka + 1 ordinal + 7 one-hot (4 metode bayar + 3 perangkat) + 1 biner = 13 kolom.")
    print("   Total kolom:", hasil.shape[1])
except Exception as e:
    print("❌ Belum pas:", e)

### ✏️ Tugas 2.5 — Latih Logistic Regression & lihat laporannya

Buat `model_churn` = `Pipeline` berisi `preprocessor2` dan `LogisticRegression(max_iter=1000)`. Latih, prediksi `X_test`, lalu hitung `acc` (akurasi) dan `f1` (F1-score).

In [ ]:
model_churn = Pipeline([
    ("prep", ___),
    ("clf", LogisticRegression(max_iter=1000)),
])
model_churn.fit(___, ___)

pred = model_churn.predict(___)
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
print(f"Akurasi: {acc:.3f} | F1: {f1:.3f}")
print(classification_report(y_test, pred, target_names=["Bertahan", "Churn"]))

In [ ]:
# ✅ Cek jawabanmu
try:
    assert acc > 0.75, f"Akurasi baru {acc:.3f}. Cek lagi pipeline-mu."
    print(f"✅ Oke! Akurasi {acc:.3f}. Pipeline-mu jalan dari data mentah sampai prediksi.")
except Exception as e:
    print("❌ Belum pas:", e)

### ✏️ Tugas 2.6 — Tuning: cari pengaturan terbaik dengan GridSearchCV 🔍

Keajaiban `Pipeline` + `ColumnTransformer`: kita bisa *tuning* **langkah preprocessing** dan **model** sekaligus!

Aturan nama parameter: **`namaLangkah__namaBagian__namaParameter`** (pakai dua garis bawah).

Contoh: strategi imputer angka ada di `prep__angka__imputer__strategy`.

Isi `param_grid` untuk mencoba:
- strategi imputer angka: `"mean"` atau `"median"`
- nilai `C` pada model: `0.1`, `1`, `10`

<details><summary>💡 Petunjuk (klik kalau mentok)</summary>

`"angka"` di tengah nama parameter itu adalah **nama transformer** yang kamu tulis di ColumnTransformer. Itulah kenapa nama yang jelas itu penting!

</details>

In [ ]:
param_grid = {
    "prep__angka__imputer__strategy": [___, ___],
    "clf__C": [___, ___, ___],
}

grid = GridSearchCV(model_churn, param_grid, cv=5, scoring="f1")
grid.fit(X_train, y_train)

print("Pengaturan terbaik:", grid.best_params_)
print(f"F1 (cross-validation): {grid.best_score_:.3f}")

In [ ]:
# ✅ Cek jawabanmu
try:
    assert set(grid.best_params_.keys()) == {"prep__angka__imputer__strategy", "clf__C"}, "Parameter grid belum sesuai"
    print("✅ Mantap! Kamu baru saja tuning preprocessing + model dalam satu langkah.")
    print("   Parameter terbaik:", grid.best_params_)
except Exception as e:
    print("❌ Belum pas:", e)

---
# Bagian D — Kunci Jawaban 🔑

> ⚠️ **Stop dulu!** Sudah coba kerjakan sendiri? Kalau sudah, silakan lanjut. Mengintip kunci jawaban setelah berusaha itu **bukan curang**, itu belajar. 😉

Kunci di bawah **harus dijalankan berurutan** per studi kasus (karena variabel saling berkaitan). Kalau mau menjalankan kunci jawaban, jalankan **semua sel kode dari atas** ke bawah di tiap kasus, atau jalankan ulang bagian data dulu.

## 🔑 Kunci Studi Kasus 1 — Harga Rumah 🏠

**Tugas 1.1 — Pisahkan fitur & target, lalu split data**

In [ ]:
X = df_rumah.drop(columns="harga_juta")
y = df_rumah["harga_juta"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Tugas 1.2 — Kelompokkan kolom**

In [ ]:
kolom_angka = ["luas_tanah", "luas_bangunan", "jumlah_kamar", "umur_rumah"]
kolom_kategori = ["lokasi", "kondisi"]

**Tugas 1.3 — Buat 'jalur' (pipeline) untuk tiap jenis kolom**

In [ ]:
pipe_angka = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

pipe_kategori = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

**Tugas 1.4 — Gabungkan dengan ColumnTransformer 🎯**

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("angka", pipe_angka, kolom_angka),
        ("kategori", pipe_kategori, kolom_kategori),
    ]
)
preprocessor.fit(X_train)

**Tugas 1.5 — Latih model dan ukur hasilnya**

In [ ]:
model_rumah = Pipeline([
    ("prep", preprocessor),
    ("regresor", RandomForestRegressor(n_estimators=100, random_state=42)),
])
model_rumah.fit(X_train, y_train)

pred = model_rumah.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
print(f"MAE: {mae:.1f} juta | R2: {r2:.3f}")

## 🔑 Kunci Studi Kasus 2 — Churn Langganan 📺

**Tugas 2.1 — Siapkan data (pakai stratify!)**

In [ ]:
X = df_churn.drop(columns="churn")
y = df_churn["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Tugas 2.2 — Pilih kolom angka secara otomatis**

In [ ]:
selektor_angka = make_column_selector(dtype_include="float")
print(selektor_angka(X_train))

**Tugas 2.3 — Siapkan 3 transformer berbeda**

In [ ]:
tf_angka = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

tf_ordinal = OrdinalEncoder(categories=[["Basic", "Standard", "Premium"]])

tf_nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

**Tugas 2.4 — Rakit ColumnTransformer dengan 4 jalur**

In [ ]:
preprocessor2 = ColumnTransformer(
    transformers=[
        ("angka", tf_angka, selektor_angka),
        ("ordinal", tf_ordinal, ["paket"]),
        ("nominal", tf_nominal, ["metode_bayar", "perangkat"]),
        ("biner", "passthrough", ["punya_promo"]),
    ],
    remainder="drop",
).set_output(transform="pandas")

hasil = preprocessor2.fit_transform(X_train)
hasil.head()

**Tugas 2.5 — Latih Logistic Regression & lihat laporannya**

In [ ]:
model_churn = Pipeline([
    ("prep", preprocessor2),
    ("clf", LogisticRegression(max_iter=1000)),
])
model_churn.fit(X_train, y_train)

pred = model_churn.predict(X_test)
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
print(f"Akurasi: {acc:.3f} | F1: {f1:.3f}")
print(classification_report(y_test, pred, target_names=["Bertahan", "Churn"]))

**Tugas 2.6 — Tuning: cari pengaturan terbaik dengan GridSearchCV 🔍**

In [ ]:
param_grid = {
    "prep__angka__imputer__strategy": ["mean", "median"],
    "clf__C": [0.1, 1, 10],
}

grid = GridSearchCV(model_churn, param_grid, cv=5, scoring="f1")
grid.fit(X_train, y_train)

print("Pengaturan terbaik:", grid.best_params_)
print(f"F1 (cross-validation): {grid.best_score_:.3f}")

---
# Bagian E — Klaim Badge 🏆🦊

Selamat! Kalau kamu sudah menyelesaikan:
- ✅ Kuis teori Bagian A
- ✅ Studi Kasus 1 dan 2 (semua sel "Cek jawabanmu" muncul ✅)

maka kamu resmi mendapatkan badge **"ColumnTransformer Master"** 🦊🏆 (file `badge_columntransformer.png`).

**Cara klaim:** screenshot hasil akhir notebook-mu (atau tangkapan layar semua centang ✅), pasang badge-nya di profilmu, lalu tag kreatornya sambil ceritakan satu hal baru yang kamu pelajari!

### 🎓 Rangkuman Super Singkat
```python
ColumnTransformer(
    transformers=[
        ("nama_jalur", transformer_atau_pipeline, kolom),
        ...
    ],
    remainder="drop",   # atau "passthrough"
)
```
- 🧩 **Paralel**: tiap kelompok kolom punya jalurnya sendiri.
- 🔗 Masukkan ke `Pipeline` bersama model → **bebas data leakage**.
- 📛 `get_feature_names_out()` → lihat nama kolom hasil.
- ⚠️ Teks → nama kolom **string**; angka/kategori → **list**.

**Sampai jumpa di topik berikutnya! 🦊✨**